# Poster: main project pipeline (compact)

**Top-to-bottom** pipeline (portrait): **Datasets → Preprocessing → Training → Analysis**. Arrows run **downward**. Title case stage names; body copy is larger and hierarchically typeset.

The figure is **taller than wide** (`figheight > figwidth`) to match vertical poster space; full canvas is saved (no `bbox_inches='tight'` crop).

- `reports/figures/poster_pipeline_overview.png`
- `reports/figures/poster_pipeline_overview.pdf`


In [8]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyBboxPatch, Polygon

DARK = "#3a1f56"
FILL = "#e8dff5"
ARROW = DARK

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif", "Bitstream Vera Serif"],
    "font.size": 18,
    "text.color": DARK,
})

OUT_DIR = Path("../reports/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PNG = OUT_DIR / "poster_pipeline_overview.png"
OUT_PDF = OUT_DIR / "poster_pipeline_overview.pdf"

# Portrait figure (taller than wide) for vertical poster layout
fig_w, fig_h = 7.0, 11.5
fig = plt.figure(figsize=(fig_w, fig_h), dpi=150)
fig.patch.set_facecolor("white")
tfig = fig.transFigure

n = 4
x0 = 0.10
box_w = 0.80
top_m, bot_m = 0.048, 0.048
gap_y = 0.052
avail = 1.0 - top_m - bot_m - (n - 1) * gap_y
box_h = avail / n

stages = [
    {
        "title": "Datasets",
        "body": "PatchCamelyon (PCam)\nCamelyon17 (WILDS)",
    },
    {
        "title": "Preprocessing",
        "body": (
            "Leakage Control\n"
            "Quality Gate\n"
            "Macenko Stain Normalization\n"
            "Pixels Scaled to [0, 1]"
        ),
        "linespacing": 1.52,
        "body_y": 0.70,
    },
    {
        "title": "Training",
        "hero": "CNN vs Virchow2",
        "body": (
            "In-Domain and Out-of-Domain\n"
            "Cross-Dataset (Two Directions)"
        ),
    },
    {
        "title": "Analysis",
        "body": (
            "Performance,\n"
            "Calibration, and Uncertainty\n"
            "Bootstrap CIs and\n"
            "Permutation Tests (BH-FDR)\n"
            "Qualitative Patch Review"
        ),
        "body_center_band": True,
        "body_band_top_frac": 0.64,
        "body_band_bot_frac": 0.10,
        "linespacing": 1.38,
    },
]

title_fs = 28
hero_fs = 24
body_fs = 19

# ys[i] = lower-left y of box i; stage 0 at top (high y)
ys = []
y = 1.0 - top_m - box_h
for i in range(n):
    ys.append(y)
    y -= gap_y + box_h

xc = x0 + box_w / 2

for y_ll, st in zip(ys, stages):
    box = FancyBboxPatch(
        (x0, y_ll),
        box_w,
        box_h,
        boxstyle="round,pad=0.006,rounding_size=0.01",
        linewidth=2.4,
        edgecolor=DARK,
        facecolor=FILL,
        transform=tfig,
        zorder=1,
    )
    fig.add_artist(box)
    fig.text(
        xc,
        y_ll + box_h * 0.91,
        st["title"],
        ha="center",
        va="top",
        fontsize=title_fs,
        fontweight="bold",
        color=DARK,
        transform=tfig,
        zorder=2,
    )
    if st.get("hero"):
        fig.text(
            xc,
            y_ll + box_h * 0.62,
            st["hero"],
            ha="center",
            va="top",
            fontsize=hero_fs,
            fontweight="normal",
            color=DARK,
            transform=tfig,
            zorder=2,
        )
        fig.text(
            xc,
            y_ll + box_h * 0.34,
            st["body"],
            ha="center",
            va="top",
            fontsize=body_fs,
            color=DARK,
            linespacing=1.28,
            transform=tfig,
            zorder=2,
        )
    else:
        ls = float(st.get("linespacing", 1.22))
        if st.get("body_center_band"):
            y_hi = y_ll + box_h * float(st.get("body_band_top_frac", 0.82))
            y_lo = y_ll + box_h * float(st.get("body_band_bot_frac", 0.10))
            y_c = 0.5 * (y_hi + y_lo)
            fig.text(
                xc,
                y_c,
                st["body"],
                ha="center",
                va="center",
                fontsize=body_fs,
                color=DARK,
                linespacing=ls,
                transform=tfig,
                zorder=2,
            )
        else:
            body_y = float(st.get("body_y", 0.52))
            fig.text(
                xc,
                y_ll + box_h * body_y,
                st["body"],
                ha="center",
                va="top",
                fontsize=body_fs,
                color=DARK,
                linespacing=ls,
                transform=tfig,
                zorder=2,
            )

def _gap_arrow_down(fig, xc, y_upper_bottom, gap, color, lw=3.4):
    """Gap lies below upper box; bottom of upper = y_upper_bottom (high y)."""
    tip_y = y_upper_bottom - gap + 0.004
    head = 0.020 + 0.35 * gap
    base_y = tip_y + head
    shaft_top = y_upper_bottom - 0.006
    if base_y >= shaft_top:
        base_y = shaft_top - 0.014
    fig.add_artist(
        Line2D(
            [xc, xc],
            [shaft_top, base_y],
            transform=fig.transFigure,
            color=color,
            lw=lw,
            solid_capstyle="round",
            zorder=5,
            clip_on=False,
        )
    )
    hw = 0.016 + 0.25 * gap
    tri = Polygon(
        [[xc - hw, base_y], [xc + hw, base_y], [xc, tip_y]],
        closed=True,
        transform=fig.transFigure,
        facecolor=color,
        edgecolor=color,
        lw=0.4,
        zorder=6,
        clip_on=False,
    )
    fig.add_artist(tri)


for i in range(n - 1):
    _gap_arrow_down(fig, xc, ys[i], gap_y, ARROW)

fig.subplots_adjust(0, 0, 1, 1)
fig.savefig(OUT_PNG, dpi=150, facecolor="white")
fig.savefig(OUT_PDF, facecolor="white")
plt.show()
print("Wrote:", OUT_PNG.resolve())
print("Wrote:", OUT_PDF.resolve())


<Figure size 1050x1725 with 0 Axes>

Wrote: C:\GP_ECG\reports\figures\poster_pipeline_overview.png
Wrote: C:\GP_ECG\reports\figures\poster_pipeline_overview.pdf
